# v3.6 Phase 7: Security Audit Notebook

This notebook runs the v3.6 security checks:
1. No binary artifacts in git
2. No token logging (redaction check)
3. Non-commercial models excluded from default-safe core
4. LocateAnything license guard active

In [ ]:
import subprocess

# Check 1: No binary artifacts in git
result = subprocess.run(['git', 'ls-files'], capture_output=True, text=True)
files = result.stdout.splitlines()
weight_exts = {'.onnx', '.pt', '.pth', '.ckpt', '.safetensors', '.engine', '.trt', '.bin'}
bad = [f for f in files if any(f.endswith(e) for e in weight_exts)]

if bad:
    print(f'FAIL: Binary artifacts in git: {bad}')
else:
    print('PASS: No binary artifacts in git')

In [ ]:
# Check 2: Token redaction
from visionservex.cli.sam_commands import _redact

test_token = 'hf_abcdefghij1234'
redacted = _redact(test_token)
print(f'Original: {test_token}')
print(f'Redacted: {redacted}')

assert redacted == 'hf_***34', f'Wrong redaction format: {redacted}'
assert test_token not in redacted
print('PASS: Token redaction works correctly')

In [ ]:
# Check 3: Non-commercial models excluded from default-safe core
from visionservex.vsx import VSX, _LOCATEANYTHING_FACTS

for mid in _LOCATEANYTHING_FACTS['_model_ids'].split():
    info = VSX.locateanything(mid).explain()
    assert info['default_safe'] is False, f'{mid}: default_safe should be False'
    assert info['commercial_safe'] is False, f'{mid}: commercial_safe should be False'

print('PASS: All LocateAnything models excluded from default-safe core')

In [ ]:
# Check 4: LocateAnything license guard
from PIL import Image
from visionservex.vsx import VSX, VSXError

img = Image.new('RGB', (32, 32))
h = VSX.locateanything('locate-anything-3b')
try:
    h.locate(img, text='cat', accept_noncommercial=False)
    print('FAIL: Should have raised VSXError')
except VSXError:
    print('PASS: License guard prevents execution without accept_noncommercial=True')

In [ ]:
# Summary
print('\nv3.6 Security Audit Summary:')
print('  [PASS] No binary artifacts in git')
print('  [PASS] Token redaction format: first 3 + *** + last 2')
print('  [PASS] LocateAnything models: default_safe=False, commercial_safe=False')
print('  [PASS] License guard blocks inference without explicit acknowledgment')